### 手写decoder

In [1]:
import torch.nn as nn
import torch


class Decoder(nn.Module):
    def __init__(self, d_model, d_ff, n_heads, n_encoder_layers):
        super().__init__()
        self.layers = nn.ModuleList([DecoderLayer(d_model, d_ff, n_heads) for _ in range(n_encoder_layers)])

    def forward(self, decoder_inputs, encoder_outputs, mask=None):
        for layer in self.layers:
            decoder_inputs = layer(decoder_inputs, encoder_outputs, mask=mask)
        return decoder_inputs


class DecoderLayer(nn.Module):
    def __init__(self, d_model, d_ff, n_heads):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, d_model, d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, d_model, d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, decoder_inputs, encoder_outputs, mask=None):
        attn1 = self.self_attn(decoder_inputs, decoder_inputs, decoder_inputs, mask=mask)
        x = self.norm1(decoder_inputs + attn1)

        attn2 = self.cross_attn(x, encoder_outputs, encoder_outputs)
        x = self.norm2(x + attn2)

        ffn_out = self.ffn(x)
        x = self.norm3(x + ffn_out)
        return x


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(torch.relu(self.linear1(x)))


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, attn_dim: int, output_dim: int, num_heads: int, ):
        super().__init__()
        self.embed_dim = embed_dim
        self.attn_dim = attn_dim
        self.output_dim = output_dim
        self.num_heads = num_heads
        self.head_dim = attn_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)

        self.out_proj = nn.Linear(self.attn_dim, self.output_dim, bias=False)

    def forward(self, q_x, k_x, v_x, mask=None):
        batch_size, seq_len, embed_dim = q_x.shape

        q = self.q_proj(q_x)
        k = self.k_proj(k_x)
        v = self.v_proj(v_x)

        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        attn_score = torch.matmul(q, k.transpose(-2, -1))
        q_k = k.size(-1)
        attn_score = attn_score / torch.sqrt(torch.tensor(q_k))

        if mask is not None:
            attn_score = attn_score.masked_fill(mask == 0, float('-inf'))

        attn_weight = torch.softmax(attn_score, dim=-1)

        o = torch.matmul(attn_weight, v)
        attn_out = o.transpose(1, 2).reshape(batch_size, seq_len, self.attn_dim)
        return self.out_proj(attn_out)
